# Logos — 1 B MoE Architecture Bake-off

Pre-train **seven** decoder-only transformer variants on WikiText-103 with the same data, the same MoE FFN, and a matched parameter budget — then compare their loss curves side-by-side.

| Variant       | Architecture                                                                                       | Parameters (~) |
|---------------|----------------------------------------------------------------------------------------------------|----------------|
| `baseline`    | Rotary MHA + MoE FFN.                                                                              | ~1.00 B        |
| `linear`      | Kimi Delta Attention (chunkwise / O(1) recurrent) replaces softmax attention.                      | ~1.01 B        |
| `recursive`   | Entry / Body / Exit with shared body weights applied multiple times per forward.                   | ~1.00 B        |
| `residual`    | Block Attention Residuals replace the additive residual at every sublayer.                         | ~1.00 B        |
| `superlinear` | KDA with MLA-compressed state snapshots and sparse top-k retrieval attention.                      | ~1.03 B        |
| `hybrid`      | KDA + retrieval with local sliding-window softmax layers interleaved.                              | ~1.02 B        |
| `logos`       | Hybrid attention in an Entry / Body / Exit looped-depth structure with Block Attention Residuals.  | ~1.03 B        |

All seven use **`d_model=1024`**, **16 heads**, 16 transformer layers (or 16 unique parameter-blocks for `recursive` and `logos`), and the same MoE FFN (**2 shared + 64 sparse experts, top-k = 4, expert_d_ff = 256** — ≈ 6.25 % sparse activation). Sequence 1024, batch 4, 3 epochs, 20 000 training chunks, LR 3e-4, bf16 + `torch.compile`.

`recursive` and `logos` use 4 + 8 + 4 = 16 unique parameter-blocks with the body looped twice (24 effective layers at the same parameter count as a 16-layer model). MLA-bottlenecked retrieval keeps the memory path under ~1.3 M per layer; Block Attention Residuals add only an RMSNorm + (`d_model`, 1) projection per sublayer. Every training cell prints the exact parameter count at startup.

Every cell passes **`--no-save`**, which skips writing `checkpoint_*.pt` files. A small `history.json` is still written per model and is what the comparison cell reads. Drop `--no-save` to keep weights.


## 1. Verify the GPU is attached

In [ ]:
!nvidia-smi

## 2. Clone the repo

In [ ]:
import pathlib

REPO_URL = "https://github.com/Rorical/Logos.git"  # adjust if your fork lives elsewhere
REPO_DIR = pathlib.Path("Logos")

if REPO_DIR.exists():
    print(f"{REPO_DIR} already cloned — pulling latest")
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!ls

## 3. Install `uv` and sync project dependencies

In [ ]:
!pip install --quiet uv

In [ ]:
!uv sync

In [ ]:
!uv run python -c "import torch; print('torch:', torch.__version__); print('cuda available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')"


## 4. Pre-download WikiText-103

In [ ]:
!uv run python -c "from datasets import load_dataset; ds = load_dataset('wikitext', 'wikitext-103-raw-v1'); print({k: len(v) for k, v in ds.items()})"


## 5. Shared training settings

Every training cell below uses these flags:

```
--dataset wikitext --dataset-config wikitext-103-raw-v1
--d-model 1024 --num-heads 16
--use-moe --num-shared-experts 2 --num-sparse-experts 64 --top-k 4 --expert-d-ff 256
--max-length 1024 --batch-size 4
--epochs 3 --lr 3e-4 --warmup-steps 200 --weight-decay 0.1
--max-train-examples 20000 --max-val-examples 500
--num-workers 2
--device cuda --bf16 --compile --compile-mode default
--sample-every 1 --sample-prompt "The history of linear algebra"
--no-save
```

Only the model-specific flags (`--model`, architectural knobs) differ between cells.


## 6. Train `baseline` (~1.00 B params)

Reference architecture: 16 standard Transformer layers with rotary multi-head self-attention and the shared MoE FFN.


In [ ]:
!uv run python scripts/train.py \
    --model baseline \
    --num-layers 16 \
    --dataset wikitext --dataset-config wikitext-103-raw-v1 \
    --d-model 1024 --num-heads 16 \
    --use-moe --num-shared-experts 2 --num-sparse-experts 64 \
        --top-k 4 --expert-d-ff 256 \
    --max-length 1024 --batch-size 4 \
    --epochs 3 --lr 3e-4 --warmup-steps 200 --weight-decay 0.1 \
    --max-train-examples 20000 --max-val-examples 500 \
    --num-workers 2 \
    --device cuda --bf16 --compile --compile-mode default \
    --sample-every 1 --sample-prompt "The history of linear algebra" \
    --no-save --save-dir runs/baseline


## 7. Train `linear` (~1.01 B params)

Kimi Delta Attention replaces softmax attention. Train cost is comparable at this sequence length; inference is O(1) per token via the recurrent form.


In [ ]:
!uv run python scripts/train.py \
    --model linear \
    --num-layers 16 --head-dim 64 --chunk-size 64 --conv-size 4 \
    --dataset wikitext --dataset-config wikitext-103-raw-v1 \
    --d-model 1024 --num-heads 16 \
    --use-moe --num-shared-experts 2 --num-sparse-experts 64 \
        --top-k 4 --expert-d-ff 256 \
    --max-length 1024 --batch-size 4 \
    --epochs 3 --lr 3e-4 --warmup-steps 200 --weight-decay 0.1 \
    --max-train-examples 20000 --max-val-examples 500 \
    --num-workers 2 \
    --device cuda --bf16 --compile --compile-mode default \
    --sample-every 1 --sample-prompt "The history of linear algebra" \
    --no-save --save-dir runs/linear


## 8. Train `recursive` (~1.00 B params)

Entry / Body / Exit looped-depth architecture: 4 entry + 8 body + 4 exit = 16 unique parameter-blocks, with the body applied twice per forward (24 effective layers at the same parameter count). Per-channel injection gates `A`, `B` start at zero.


In [ ]:
!uv run python scripts/train.py \
    --model recursive \
    --num-entry-layers 4 --num-body-layers 8 --num-exit-layers 4 \
    --num-loops 2 \
    --dataset wikitext --dataset-config wikitext-103-raw-v1 \
    --d-model 1024 --num-heads 16 \
    --use-moe --num-shared-experts 2 --num-sparse-experts 64 \
        --top-k 4 --expert-d-ff 256 \
    --max-length 1024 --batch-size 4 \
    --epochs 3 --lr 3e-4 --warmup-steps 200 --weight-decay 0.1 \
    --max-train-examples 20000 --max-val-examples 500 \
    --num-workers 2 \
    --device cuda --bf16 --compile --compile-mode default \
    --sample-every 1 --sample-prompt "The history of linear algebra" \
    --no-save --save-dir runs/recursive


## 9. Train `residual` (~1.00 B params)

Standard 16-layer model with **Block Attention Residuals**: every sublayer's input is computed by a learned softmax over completed-block representations and the running partial sum. Zero-initialised projections make training begin identical to `baseline`.


In [ ]:
!uv run python scripts/train.py \
    --model residual \
    --num-layers 16 --num-blocks 4 \
    --dataset wikitext --dataset-config wikitext-103-raw-v1 \
    --d-model 1024 --num-heads 16 \
    --use-moe --num-shared-experts 2 --num-sparse-experts 64 \
        --top-k 4 --expert-d-ff 256 \
    --max-length 1024 --batch-size 4 \
    --epochs 3 --lr 3e-4 --warmup-steps 200 --weight-decay 0.1 \
    --max-train-examples 20000 --max-val-examples 500 \
    --num-workers 2 \
    --device cuda --bf16 --compile --compile-mode default \
    --sample-every 1 --sample-prompt "The history of linear algebra" \
    --no-save --save-dir runs/residual


## 10. Train `superlinear` (~1.03 B params)

Kimi Delta Attention augmented with **MLA-compressed state snapshots** (every 256 tokens) and a **sparse top-k retrieval** sublayer over those snapshots. Retrieval `out_up` is zero-initialised so training begins identical to `linear`.


In [ ]:
!uv run python scripts/train.py \
    --model superlinear \
    --num-layers 16 --head-dim 64 --chunk-size 64 --conv-size 4 \
    --snapshot-interval 256 --snapshot-latent-dim 128 \
    --mem-top-k 16 --mem-head-dim 64 \
    --dataset wikitext --dataset-config wikitext-103-raw-v1 \
    --d-model 1024 --num-heads 16 \
    --use-moe --num-shared-experts 2 --num-sparse-experts 64 \
        --top-k 4 --expert-d-ff 256 \
    --max-length 1024 --batch-size 4 \
    --epochs 3 --lr 3e-4 --warmup-steps 200 --weight-decay 0.1 \
    --max-train-examples 20000 --max-val-examples 500 \
    --num-workers 2 \
    --device cuda --bf16 --compile --compile-mode default \
    --sample-every 1 --sample-prompt "The history of linear algebra" \
    --no-save --save-dir runs/superlinear


## 11. Train `hybrid` (~1.02 B params)

`superlinear` with **local sliding-window softmax layers** (window 256) placed every fourth position. KDA layers retain snapshot retrieval; SWA layers carry only local exact attention. Inference state stays bounded: O(1) per KDA layer and O(W) per SWA layer.


In [ ]:
!uv run python scripts/train.py \
    --model hybrid \
    --num-layers 16 --head-dim 64 --chunk-size 64 --conv-size 4 \
    --snapshot-interval 256 --snapshot-latent-dim 128 \
    --mem-top-k 16 --mem-head-dim 64 \
    --swa-every 4 --swa-offset 3 --swa-window 256 \
    --dataset wikitext --dataset-config wikitext-103-raw-v1 \
    --d-model 1024 --num-heads 16 \
    --use-moe --num-shared-experts 2 --num-sparse-experts 64 \
        --top-k 4 --expert-d-ff 256 \
    --max-length 1024 --batch-size 4 \
    --epochs 3 --lr 3e-4 --warmup-steps 200 --weight-decay 0.1 \
    --max-train-examples 20000 --max-val-examples 500 \
    --num-workers 2 \
    --device cuda --bf16 --compile --compile-mode default \
    --sample-every 1 --sample-prompt "The history of linear algebra" \
    --no-save --save-dir runs/hybrid


## 12. Train `logos` (~1.03 B params)

Hybrid attention (KDA + retrieval with SWA layers interleaved) inside an Entry / Body / Exit looped-depth structure, with Block Attention Residuals at every sublayer. Body MoE layers carry per-loop router-bias rows and a cross-loop expert-diversity term.


In [ ]:
!uv run python scripts/train.py \
    --model logos \
    --num-entry-layers 4 --num-body-layers 8 --num-exit-layers 4 \
    --num-loops 2 \
    --head-dim 64 --chunk-size 64 --conv-size 4 \
    --snapshot-interval 256 --snapshot-latent-dim 128 \
    --mem-top-k 16 --mem-head-dim 64 \
    --swa-every 4 --swa-offset 3 --swa-window 256 \
    --moe-diversity-factor 0.5 \
    --dataset wikitext --dataset-config wikitext-103-raw-v1 \
    --d-model 1024 --num-heads 16 \
    --use-moe --num-shared-experts 2 --num-sparse-experts 64 \
        --top-k 4 --expert-d-ff 256 \
    --max-length 1024 --batch-size 4 \
    --epochs 3 --lr 3e-4 --warmup-steps 200 --weight-decay 0.1 \
    --max-train-examples 20000 --max-val-examples 500 \
    --num-workers 2 \
    --device cuda --bf16 --compile --compile-mode default \
    --sample-every 1 --sample-prompt "The history of linear algebra" \
    --no-save --save-dir runs/logos


## 13. Head-to-head comparison

Loads every model's `runs/<name>/history.json` (per-epoch train / val loss, perplexity, time) and prints a side-by-side table plus a loss-curve plot. Skips any model whose history file isn't on disk, so this cell can be re-run mid-sweep.


In [ ]:
import json, pathlib

CANONICAL = ['baseline', 'linear', 'recursive', 'residual', 'superlinear', 'hybrid', 'logos']
runs_root = pathlib.Path('runs')
found = sorted(p.parent.name for p in runs_root.glob('*/history.json')) if runs_root.exists() else []
ordered = [n for n in CANONICAL if n in found] + [n for n in found if n not in CANONICAL]

results = {}
for name in ordered:
    path = runs_root / name / 'history.json'
    with path.open() as f:
        data = json.load(f)
    results[name] = data
    print(f"[loaded] {name}: {len(data['history'])} epochs")
if not results:
    print('No runs/*/history.json found — train at least one model first.')

if results:
    print()
    header = f'{"model":<16}{"epoch":>6}{"train_loss":>12}{"train_ppl":>12}{"val_loss":>12}{"val_ppl":>12}{"time_s":>10}'
    print(header)
    print('-' * len(header))
    for name, r in results.items():
        for h in r['history']:
            tr = h['train']
            val = h.get('val') or {}
            vl = val.get('loss', float('nan'))
            vp = val.get('ppl', float('nan'))
            print(f"{name:<16}{h['epoch']:>6}{tr['loss']:>12.4f}{tr['ppl']:>12.2f}"
                  f"{vl:>12.4f}{vp:>12.2f}{h.get('time', 0):>10.1f}")

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 5))
    for name, r in results.items():
        epochs = [h['epoch'] for h in r['history']]
        if r['history'] and r['history'][0].get('val'):
            ys = [h['val']['loss'] for h in r['history']]
            lbl = f'{name} (val)'
        else:
            ys = [h['train']['loss'] for h in r['history']]
            lbl = f'{name} (train)'
        ax.plot(epochs, ys, marker='o', label=lbl)
    ax.set_xlabel('epoch')
    ax.set_ylabel('loss')
    ax.set_title('Architecture bake-off — 1 B MoE on WikiText-103')
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print('(matplotlib not installed — install with `pip install matplotlib` for a plot)')


## 14. Peek at GPU memory

In [ ]:
!nvidia-smi

## 15. Download training histories

A small `histories.tar.gz` is enough to inspect the bake-off offline (no model weights). The `runs/` directory only carries `history.json` files when `--no-save` is set.


In [ ]:
import shutil, pathlib

if pathlib.Path('runs').exists():
    shutil.make_archive('histories', 'gztar', '.', 'runs')
    from google.colab import files
    files.download('histories.tar.gz')
else:
    print('No runs/ directory yet — train at least one model first.')
